# 3ScoringVADER

# Sentiment Analysis dengan VADER

Notebook ini mengganti metode FinBERT dengan **VADER (Valence Aware Dictionary and sEntiment Reasoner)** dari library `vaderSentiment`.

Karena data berbahasa Indonesia, kita tambahkan **translasi otomatis** menggunakan `googletrans` sebelum VADER dijalankan.

In [ ]:
# ==========================
# INSTALL DEPENDENCIES
# ==========================
!pip install vaderSentiment googletrans==4.0.0-rc1 tqdm --quiet

In [ ]:
import gdown

files = {
    'detikv2_2016.csv': '1ZkRSFCWHnMXbUgvcP5HiDNhF7sQle5jN',
}
'''
files = {
    'detikv2_2022.csv': '18AzOZrjDrwU0gbn_SXOsHcIw7XQUFWCg',
}
'''
for name, file_id in files.items():
    print(f'Downloading {file_id} to {name}')
    gdown.download(id=file_id, output=name, quiet=False)


# ==========================
# LOAD DATA
# ==========================
import pandas as pd

files = ["detikv2_2016.csv"]
df_list = [pd.read_csv(f, on_bad_lines='skip', encoding='utf-8') for f in files]
df_full = pd.concat(df_list, ignore_index=True)

print(f"Total artikel sebelum deduplikasi: {len(df_full)}")

Downloading...
From: https://drive.google.com/uc?id=1ZkRSFCWHnMXbUgvcP5HiDNhF7sQle5jN
To: /kaggle/working/detikv2_2016.csv
100%|██████████| 43.9M/43.9M [00:00<00:00, 68.4MB/s]


Total artikel sebelum deduplikasi: 22539


In [ ]:
# ==========================
# HAPUS DUPLIKAT
# (berdasarkan isi artikel — judul bisa berbeda tapi isi sama)
# ==========================
kolom_teks = "isi"

# Normalisasi whitespace dulu sebelum cek duplikat
# agar "teks  sama" dan "teks sama" dianggap sama
df_full['isi_normalized'] = (
    df_full[kolom_teks]
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)

n_before = len(df_full)
df_full = df_full.drop_duplicates(subset='isi_normalized', keep='first').copy()
n_after = len(df_full)
print(f"Duplikat dihapus: {n_before - n_after} artikel")
print(f"Total artikel setelah deduplikasi: {n_after}")

# Hapus kolom bantu normalisasi
df_full.drop(columns=['isi_normalized'], inplace=True)

Duplikat dihapus: 1 artikel
Total artikel setelah deduplikasi: 22538


In [ ]:
# ==========================
# FILTER DATA
# ==========================

import re
keywords = [
  "harga minyak",
  "transisi energi",
  "perubahan iklim",
  "emisi karbon",
  "tarif listrik",
  "karbon",
  "tenaga surya",
  "kementerian energi",
  "net zero emission",
  "pln",
  "nze",
  "bioenergi",
  "jaringan listrik",
  "crude oil",
  "energi hijau",
  "minyak mentah",
  "turbin angin",
  "batubara"
]

pattern = '|'.join([re.escape(k) for k in keywords])
df_filter = df_full[
    df_full[kolom_teks].str.contains(pattern, case=False, na=False)
].copy()

print(f"\nJumlah data setelah filter kata kunci: {len(df_filter)} dari {len(df_full)}")
print(f"Persentase artikel relevan: {len(df_filter)/len(df_full)*100:.2f}%")
print(df_filter.head())


Jumlah data setelah filter kata kunci: 2860 dari 22538
Persentase artikel relevan: 12.69%
    id     tanggal                                              judul  \
2    3  2016-01-01  Resmikan Pabrik Sagu Terbesar, Jokowi: Semoga ...   
3    4  2016-01-01  Harga Minyak Diprediksi Masih Loyo di 2016, Pe...   
8    9  2016-01-01  Dahsyat, Ini Bos di Bawah 40 Tahun dengan Baya...   
13  14  2016-01-02  Listrik Padam di Bandara Kualanamu, Ini Penjel...   
17  18  2016-01-02      Awal Tahun, Tarif Listrik dan Harga BBM Turun   

                                                  isi  
2   Kais - Presiden Joko Widodo (Jokowi) pada Juma...  
3   Jakarta - Banyak pihak memprediksi harga minya...  
8   Jakarta - Umur belum 40 tahun namun pendapatan...  
13  Jakarta - Pasokan listrik ke Bandara Kualanamu...  
17  Jakarta - Awal tahun ini, masyarakat seperti m...  


In [ ]:
# ==========================
# CLEANING
# (dioptimalkan untuk teks berita Indonesia agar sent_tokenize akurat)
# ==========================
import re
import unicodedata

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKD", text)
    text = re.sub("\\xad", "", text)

    # --- Singkatan umum berita Indonesia ---
    # Ganti sebelum sent_tokenize agar titik di singkatan tidak
    # dianggap akhir kalimat
    text = re.sub(" Tbk\\.", " Tbk ", text)
    text = re.sub(" Tbk", " Tbk ", text)
    text = re.sub(" Rp\\.", " ", text)
    text = re.sub(" Rp", " ", text)
    text = re.sub(" PT\\.", " PT ", text)
    text = re.sub(" Pt\\.", " PT ", text)
    text = re.sub(" dr\\.", " ", text)
    text = re.sub(" Dr\\.", " ", text)
    text = re.sub(" DR\\.", " ", text)
    text = re.sub(" N\\.A\\.", " ", text)
    text = re.sub(" H\\.M\\.", " HM", text)
    text = re.sub(" jl\\. ", " jalan ", text)
    text = re.sub(" Jl\\. ", " jalan ", text)
    text = re.sub(" Jln\\. ", " jalan ", text)
    text = re.sub(" jln\\. ", " jalan ", text)

    # --- URL ---
    text = re.sub(
        r'''(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:'".,<>?«»\u201c\u201d\u2018\u2019]))''',
        "link-website", text
    )

    # --- Pola angka & format ---
    text = re.sub(r"\. *[0-9]+\. ", ". ", text)       # hapus nomor list: 1. 2.
    text = re.sub(r": *[0-9]+\. ", ". ", text)        # hapus nomor list setelah :
    text = re.sub("(?<=\\d)(?=[a-zA-Z])", " ", text)  # pisahkan angka-huruf
    text = re.sub("(?<=[a-zA-Z])(?=\\d)", " ", text)  # pisahkan huruf-angka
    text = re.sub(" ([a-zA-Z]{1,2})\\. ", " ", text)  # hapus singkatan 1-2 huruf
    text = re.sub(r"(?<=\d)[,.](?=\d)", "", text)     # hapus titik/koma dalam angka

    # --- Normalisasi spasi & tanda baca ---
    text = re.sub("\\*", "", text)
    text = re.sub(":", ": ", text)
    text = re.sub(r"\.(?=\S)", ". ", text)
    text = re.sub(r"\?(?=\S)", "? ", text)
    text = re.sub(r"\)(?=\S)", ") ", text)
    text = re.sub(r"\((?=\S)", " (", text)
    text = re.sub("\\. +\\.", ".", text)
    text = re.sub(" +\\? +", "? ", text)
    text = re.sub(" +,", ", ", text)
    text = re.sub(" +\\. ", ". ", text)
    text = re.sub("\\.,", ", ", text)
    text = re.sub(" {2,}", " ", text)
    text = re.sub("\\.{2,}", ".", text)
    text = re.sub("\\?{2,}", ".", text)
    text = re.sub(r'\. *\([^)]*\)[\. ]', ". ", text)  # hapus kurung awal kalimat
    text = re.sub(r'\. *\([^)]*\)$', ".", text)       # hapus kurung akhir artikel

    return text.strip()

df_filter['isi_clean'] = df_filter['isi'].apply(clean_text)

# Hapus artikel yang setelah cleaning menjadi kosong atau terlalu pendek
# (kurang dari 50 karakter dianggap tidak informatif untuk analisis sentimen)
n_before_clean = len(df_filter)
df_filter = df_filter[df_filter['isi_clean'].str.len() >= 50].copy()
print(f"\nArtikel terhapus karena terlalu pendek setelah cleaning: "
      f"{n_before_clean - len(df_filter)}")
print(f"Total artikel siap translasi: {len(df_filter)}")
print(df_filter['isi_clean'].head(2))


Artikel terhapus karena terlalu pendek setelah cleaning: 0
Total artikel siap translasi: 2860
2    Kais - Presiden Joko Widodo (Jokowi) pada Juma...
3    Jakarta - Banyak pihak memprediksi harga minya...
Name: isi_clean, dtype: object


In [ ]:
# ==========================
# TRANSLASI INDONESIA -> INGGRIS
# ==========================
from googletrans import Translator
from tqdm import tqdm
import time

translator = Translator()

def translate_to_en(text, retries=5, delay=2):
    """
    Terjemahkan teks Indonesia ke Inggris untuk VADER.
    Jika gagal setelah semua percobaan, kembalikan teks asli
    sebagai fallback agar pipeline tidak terhenti.
    """
    # Validasi input
    if not isinstance(text, str) or not text.strip():
        return ""

    # Jika teks terlalu panjang, potong untuk menghindari timeout
    # Google Translate memiliki batas karakter per request
    MAX_CHARS = 4500
    if len(text) > MAX_CHARS:
        text = text[:MAX_CHARS]

    for attempt in range(retries):
        try:
            result = translator.translate(text, src='id', dest='en')

            # Validasi hasil translasi tidak kosong
            if result is None or result.text is None or not result.text.strip():
                raise ValueError("Hasil translasi kosong")

            # Validasi hasil translasi bukan karakter aneh
            # (kadang googletrans mengembalikan karakter tidak terbaca)
            translated = result.text.strip()
            if len(translated) < 5:
                raise ValueError("Hasil translasi terlalu pendek")

            return translated

        except Exception as e:
            print(f"  ⚠️ Translasi gagal (percobaan {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                # Exponential backoff: tunggu lebih lama di setiap percobaan
                wait = delay * (attempt + 1)
                time.sleep(wait)
            else:
                # Fallback: gunakan teks asli jika semua percobaan gagal
                # VADER akan memberikan skor netral untuk teks Indonesia
                # ini dilaporkan sebagai limitasi
                print(f"  ❌ Semua percobaan gagal, menggunakan teks asli sebagai fallback")
                return text

tqdm.pandas(desc="Translasi")
df_filter['isi_en'] = df_filter['isi_clean'].progress_apply(translate_to_en)

# Laporan hasil translasi
n_fallback = df_filter['isi_en'].equals(df_filter['isi_clean'])
empty_translations = (df_filter['isi_en'].str.strip() == "").sum()
print(f"\nArtikel dengan translasi kosong: {empty_translations}")
print("Translasi selesai.")
print(df_filter[['isi_clean', 'isi_en']].head(3))

Translasi: 100%|██████████| 2860/2860 [48:07<00:00,  1.01s/it] 


Artikel dengan translasi kosong: 0
Translasi selesai.
                                           isi_clean  \
2  Kais - Presiden Joko Widodo (Jokowi) pada Juma...   
3  Jakarta - Banyak pihak memprediksi harga minya...   
8  Jakarta - Umur belum 40 tahun namun pendapatan...   

                                              isi_en  
2  Kais - President Joko Widodo (Jokowi) this Fri...  
3  Jakarta - Many parties predict that world oil ...  
8  Jakarta - Not yet 40 years old, a number of co...  


In [ ]:
# ==========================
# VADER SENTIMENT SCORING
# ==========================
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

analyzer = SentimentIntensityAnalyzer()

def vader_sentiment_mean(text_en):
    """
    Hitung rata-rata skor VADER per kalimat.
    Mengembalikan dict: mean_score, mean_pos, mean_neg, mean_neu
    """
    # Validasi input
    if not isinstance(text_en, str) or not text_en.strip():
        return {
            'mean_score': 0.0,
            'mean_pos':   0.0,
            'mean_neg':   0.0,
            'mean_neu':   1.0,
            'n_sentences': 0
        }

    sentences = sent_tokenize(text_en)

    # Fallback jika tokenisasi gagal
    if not sentences:
        sentences = [text_en]

    scores = [analyzer.polarity_scores(s) for s in sentences]

    mean_score = sum(s['compound'] for s in scores) / len(scores)
    mean_pos   = sum(s['pos']      for s in scores) / len(scores)
    mean_neg   = sum(s['neg']      for s in scores) / len(scores)
    mean_neu   = sum(s['neu']      for s in scores) / len(scores)

    return {
        'mean_score':  mean_score,
        'mean_pos':    mean_pos,
        'mean_neg':    mean_neg,
        'mean_neu':    mean_neu,
        'n_sentences': len(sentences)
    }

tqdm.pandas(desc="VADER Scoring")
results = df_filter['isi_en'].progress_apply(vader_sentiment_mean)

df_filter['mean_score']  = results.apply(lambda x: x['mean_score'])
df_filter['mean_pos']    = results.apply(lambda x: x['mean_pos'])
df_filter['mean_neg']    = results.apply(lambda x: x['mean_neg'])
df_filter['mean_neu']    = results.apply(lambda x: x['mean_neu'])
df_filter['n_sentences'] = results.apply(lambda x: x['n_sentences'])

print("VADER Scoring selesai!")
print(df_filter[['tanggal', 'mean_score', 'mean_pos',
                  'mean_neg', 'mean_neu', 'n_sentences']].head())

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
VADER Scoring: 100%|██████████| 2860/2860 [00:05<00:00, 533.80it/s]

VADER Scoring selesai!
       tanggal  mean_score  mean_pos  mean_neg  mean_neu  n_sentences
2   2016-01-01    0.238783  0.042667  0.019500  0.937833            6
3   2016-01-01    0.426180  0.157400  0.019500  0.823100           10
8   2016-01-01    0.748100  0.091000  0.005000  0.903750            4
13  2016-01-02    0.016387  0.058600  0.032667  0.908800           15
17  2016-01-02    0.659875  0.086500  0.013750  0.900000            4


In [ ]:
# ==========================
# LABEL SENTIMENT OTOMATIS
# ==========================
q_pos = df_filter['mean_score'].quantile(0.67)
q_neg = df_filter['mean_score'].quantile(0.33)

print(f"Threshold positif (Q67): {q_pos:.4f}")
print(f"Threshold negatif (Q33): {q_neg:.4f}")

def label_sentiment_auto(score):
    if score >= q_pos:
        return 'positive'
    elif score <= q_neg:
        return 'negative'
    return 'neutral'

df_filter['label_sentiment'] = df_filter['mean_score'].apply(label_sentiment_auto)

print("\nDistribusi label:")
print(df_filter['label_sentiment'].value_counts())


Threshold positif (Q67): 0.3129
Threshold negatif (Q33): 0.1045

Distribusi label:
label_sentiment
neutral     972
positive    944
negative    944
Name: count, dtype: int64


In [ ]:
# ==========================
# AGREGASI HARIAN
# ==========================
df_daily = (
    df_filter.groupby('tanggal')
    .agg(
        sentiment_mean=('mean_score', 'mean'),
        pos_mean=('mean_pos',   'mean'),
        neg_mean=('mean_neg',   'mean'),
        neu_mean=('mean_neu',   'mean'),
        n_artikel=('mean_score', 'count')   # tambahan: jumlah artikel per hari
    )
    .reset_index()
)

print("Agregasi harian:")
print(df_daily.head())
print(f"\nTotal hari dengan data sentimen: {len(df_daily)}")
print(f"Rata-rata artikel per hari: {df_daily['n_artikel'].mean():.1f}")

Agregasi harian:
      tanggal  sentiment_mean  pos_mean  neg_mean  neu_mean  n_artikel
0  2016-01-01        0.471021  0.097022  0.014667  0.888228          3
1  2016-01-02        0.338131  0.072550  0.023208  0.904400          2
2  2016-01-03        0.171350  0.054833  0.017833  0.927167          2
3  2016-01-04        0.208758  0.070695  0.030285  0.899029         12
4  2016-01-05        0.027642  0.059134  0.055928  0.884913         18

Total hari dengan data sentimen: 338
Rata-rata artikel per hari: 8.5


In [ ]:
# ==========================
# SIMPAN HASIL
# ==========================
df_daily.to_csv('sentiment_vader_2016.csv', index=False)
df_filter.to_csv('sentiment_vader_full_2016.csv', index=False)

print("\nFile tersimpan:")
print("  - sentiment_vader_2016.csv       (agregasi harian)")
print("  - sentiment_vader_full_2016.csv  (per artikel)")

try:
    from IPython.display import FileLink, display
    display(FileLink('sentiment_vader_2016.csv'))
    display(FileLink('sentiment_vader_full_2016.csv'))
except:
    pass


File tersimpan:
  - sentiment_vader_2016.csv       (agregasi harian)
  - sentiment_vader_full_2016.csv  (per artikel)


/kaggle/working/sentiment_vader_2016.csv

/kaggle/working/sentiment_vader_full_2016.csv